# Importation des bibliothèques

In [1]:
import sys
!{sys.executable} -m pip install findspark


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import findspark
import os

# Indique à Python où se trouvent Spark et Java
os.environ["JAVA_HOME"] = r"C:\Users\ahoun\AppData\Local\Programs\Eclipse Adoptium\jdk-11.0.30.7-hotspot"
os.environ["SPARK_HOME"] = r"C:\spark\spark-3.5.6-bin-hadoop3"
os.environ["HADOOP_HOME"] = r"C:\hadoop"

# Initialise findspark pour faire le lien
findspark.init()

import pyspark
print("Lien vers Spark établi !")

Lien vers Spark établi !


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, when, to_date

# Constantes

In [4]:
# chemin pour accéder aux données
FILE_PATH = '../src/data/donnees_immobilieres.parquet'

# Application

In [5]:
# initialisation SparkSession
spark = SparkSession.builder.appName("RealStateProcessing").getOrCreate()

In [6]:
# chargement des données
df_raw = spark.read.parquet(FILE_PATH)

In [7]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df_raw.count(), len(df_raw.columns))

(10000, 13)

In [8]:
# Affichage du schéma
df_raw.printSchema()

root
 |-- region: string (nullable = true)
 |-- ue: string (nullable = true)
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- fonction: string (nullable = true)
 |-- designation_batiment_terrain: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- pays: string (nullable = true)
 |-- ministere: string (nullable = true)
 |-- date_inventaire: string (nullable = true)



In [9]:
# Affichage des 5 premières lignes
df_raw.show(5)

+------------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+--------------------+-----------+-----------------+------+--------------------+---------------+
|            region|                  ue|                  id|                type|            fonction|designation_batiment_terrain|dept|             adresse|code_postal|            ville|  pays|           ministere|date_inventaire|
+------------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+--------------------+-----------+-----------------+------+--------------------+---------------+
|             CORSE|a19e7cbea4c2dae3c...|936465ba5b332c604...|biens du ministèr...|biens du ministèr...|        biens du ministèr...|  2B|                NULL|       NULL|             NULL|France|Intérieur, Outre-...|        2014-03|
|NORD-PAS DE CALAIS|c1fb47969da8dd1a5...|44b7ec415ee6f810d...|bi

In [10]:
# suppression des doublons suivant l'id
df = df_raw.dropDuplicates(['id'])

In [11]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df.count(), len(df.columns))

(9041, 13)

In [12]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
df.groupBy("id").count().filter(col("count") > 1).count()

0

In [13]:
# filtrage pour ne garder que les biens en France
df = df.filter(col("pays") == "France")

In [14]:
# affichage des pays distincts
df.select("pays").distinct().show()

+------+
|  pays|
+------+
|France|
+------+



In [15]:
# conversion de la colonne "date_invnetaire" en type date
df = df.withColumn("date_inventaire", to_date(col("date_inventaire"), "yyyy-MM"))

In [16]:
# Affichage du schéma
df.printSchema()

root
 |-- region: string (nullable = true)
 |-- ue: string (nullable = true)
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- fonction: string (nullable = true)
 |-- designation_batiment_terrain: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- pays: string (nullable = true)
 |-- ministere: string (nullable = true)
 |-- date_inventaire: date (nullable = true)



In [17]:
# mise en minuscule de la colonne "ville" pour uniformiser les données
df = df.withColumn("ville", lower(col("ville")))

In [18]:
# affichage des villes distinctes
df.select("ville").distinct().show()

+--------------------+
|               ville|
+--------------------+
|          la trinité|
|               cergy|
|     gisy-les-nobles|
|   varennes-le-grand|
|    condé-sur-sarthe|
|mauzac-et-grand-c...|
|              coings|
|        la ricamarie|
|                pacé|
|     choloy-ménillot|
|              lescun|
|              mauzac|
|                reau|
|               royan|
|      pointe-à-pitre|
|            dzaoudzi|
|              bailly|
|   montigny-lès-metz|
|              lognes|
|                belz|
+--------------------+
only showing top 20 rows



In [19]:
# mise en minuscule de la colonne "region" pour uniformiser les données
df = df.withColumn("region", lower(col("region")))

In [20]:
# affichage des régions distinctes
df.select("region").distinct().show()

+--------------------+
|              region|
+--------------------+
|            bretagne|
|       ile de france|
|               c.o.m|
|            limousin|
|  nord-pas de calais|
|       franche comte|
|     basse normandie|
|            auvergne|
|               d.o.m|
|           aquitaine|
|languedoc-roussillon|
|            lorraine|
|         rhone alpes|
|      midi-pyrennees|
|              centre|
|       pays de loire|
|           bourgogne|
|  champagne-ardennes|
|                paca|
|            picardie|
+--------------------+
only showing top 20 rows



In [21]:
# anonymisation des données sensibles
df = df.withColumn("ville",when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("ville")))

In [22]:
# affichage des villes distinctes
df.select("ville").distinct().show()

+--------------------+
|               ville|
+--------------------+
|          la trinité|
|               cergy|
|     gisy-les-nobles|
|   varennes-le-grand|
|    condé-sur-sarthe|
|mauzac-et-grand-c...|
|              coings|
|        la ricamarie|
|                pacé|
|     choloy-ménillot|
|              lescun|
|              mauzac|
|                reau|
|               royan|
|      pointe-à-pitre|
|            dzaoudzi|
|              bailly|
|   montigny-lès-metz|
|              lognes|
|                belz|
+--------------------+
only showing top 20 rows



In [23]:
# anonymisation des données sensibles
df = df.withColumn("code_postal", when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("code_postal")))

In [24]:
# affichage des codes postaux distincts
df.select("code_postal").distinct().show()

+-----------+
|code_postal|
+-----------+
|      75007|
|      42370|
|      59370|
|      59169|
|      77930|
|      09120|
|      16250|
|      91910|
|      86180|
|      50800|
|      66000|
|      27130|
|      31330|
|      57655|
|      59680|
|      91120|
|      28500|
|      22130|
|      57500|
|      57180|
+-----------+
only showing top 20 rows



In [25]:
# anonymisation des données sensibles
df = df.withColumn("adresse", when(col("ministere") == "ministère de l'Intérieur", "Donnée confidentielle").otherwise(col("adresse")))

In [26]:
# affichage des adresses distinctes
df.select("adresse").distinct().show()

+--------------------+
|             adresse|
+--------------------+
|  11 Rue ELIE VERNET|
|     21 Route D AUCH|
|R DU MARECHAL LEC...|
|        29 R DELILLE|
|Lieu-dit ILET DU ...|
|     Rue DE L EGLISE|
|159  AV JACQUES D...|
|LD LE MONT FLOLENTIN|
|    LIEU-DIT MONTEAU|
|LD LA TOUR DE CHA...|
|      13 RUE GALILEE|
|  LD LA MARE POISSON|
|9  AV DOCTEUR MAU...|
|9080 ALL DE LA RO...|
|85/87  RTE DE CALAIS|
|      LD BEAU RIVAGE|
|  LD LA GRANDE GREVE|
|  LD LES GRIMPETEAUX|
|         LD KERMARIO|
|RTE DE COURCOURONNES|
+--------------------+
only showing top 20 rows



In [27]:
# uniformisation des données du ministère
df = (df
    .withColumn("ministere", when(lower(col("ministere")).like("%intérieur%"), "ministère de l'Intérieur").otherwise(lower(col("ministere"))))
    .withColumn("ministere", when(lower(col("ministere")).like("%éducation%"), "ministère de l'Éducation Nationale").otherwise(col("ministere")))
    .withColumn("ministere", when(lower(col("ministere")).like("%justice%"), "ministère de la Justice").otherwise(col("ministere")))
    .withColumn("ministere", when(lower(col("ministere")).like("%santé%"), "ministère de la Santé").otherwise(col("ministere")))
    .withColumn("ministere", when(lower(col("ministere")).like("%ecologie%") | lower(col("ministere")).like("%environnement%"), "ministère de l'Ecologie, développement et aménagement durables").otherwise(col("ministere")))
)

In [28]:
# affichage des ministères distincts
df.select("ministere").distinct().show()

+--------------------+
|           ministere|
+--------------------+
| affaires étrangères|
|ministère de l'Ec...|
|agriculture et pê...|
|      france domaine|
|               sénat|
|         agriculture|
|   premier ministre.|
|ministère de l'Éd...|
|budget, comptes p...|
|affaires étrangèr...|
|    travail - emploi|
|economie - financ...|
|enseignement supé...|
|culture et commun...|
|      conseil d'etat|
|budget - comptes ...|
|             culture|
|affaires étrangèr...|
|     multi-occupants|
|education nationa...|
+--------------------+
only showing top 20 rows



In [29]:
# Affichage du schéma
df.printSchema()

root
 |-- region: string (nullable = true)
 |-- ue: string (nullable = true)
 |-- id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- fonction: string (nullable = true)
 |-- designation_batiment_terrain: string (nullable = true)
 |-- dept: string (nullable = true)
 |-- adresse: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- pays: string (nullable = true)
 |-- ministere: string (nullable = true)
 |-- date_inventaire: date (nullable = true)



In [30]:
# Affichage des 5 premières lignes
df.show(5)

+-------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+-------+-----------+-----+------+--------------------+---------------+
|       region|                  ue|                  id|                type|            fonction|designation_batiment_terrain|dept|adresse|code_postal|ville|  pays|           ministere|date_inventaire|
+-------------+--------------------+--------------------+--------------------+--------------------+----------------------------+----+-------+-----------+-----+------+--------------------+---------------+
|ile de france|c0daef7745cdaf58f...|000d84e011f2b9e7e...|biens du ministèr...|biens du ministèr...|        biens du ministèr...|  93|   NULL|       NULL| NULL|France|ministère de l'In...|     2014-03-01|
|     limousin|1067ef9e24a6dcadc...|0020d6985224f09cd...|biens du ministèr...|biens du ministèr...|        biens du ministèr...|  23|   NULL|       NULL| NULL|France|ministère de l'In.

In [31]:
# Informations générales sur le DataFrame 
df_raw.describe().show()

+-------+-----------+--------------------+--------------------+----------------+-------------------+----------------------------+-----------------+--------------------+------------------+-----------+-----------+-------------------+---------------+
|summary|     region|                  ue|                  id|            type|           fonction|designation_batiment_terrain|             dept|             adresse|       code_postal|      ville|       pays|          ministere|date_inventaire|
+-------+-----------+--------------------+--------------------+----------------+-------------------+----------------------------+-----------------+--------------------+------------------+-----------+-----------+-------------------+---------------+
|  count|      10000|                9703|               10000|           10000|              10000|                       10000|             9998|                6712|              6985|       6972|      10000|              10000|          10000|
|   mean

In [32]:
# Affichage du nombre de valeurs non nulles par colonne et du nombre total de colonnes
(df.count(), len(df.columns))

(8883, 13)